# Phase 1 (DEAM), full-scale run -- Colab

Runs ROADMAP.md's Phase 1 for real: all 1802 DEAM clips, full-length audio,
Step 0 (TRIBEv2 -> Schaefer-400 -> windowed cache) -> Step 1 (train trunk +
VA head) -> Step 2 (held-out correlation + within-clip dynamic tracking) ->
text branch + dynamic-vs-flat sanity check (PROJECT_SPEC.md Section 6).

**Before running:**
1. Runtime -> Change runtime type -> **GPU** (T4 is fine; A100/L4 if you have
   Colab Pro and want it faster). CPU-only will technically work but Phase 0
   measured ~297s/10s-clip on CPU -- not viable at 1802 full-length clips.
2. This notebook pushes the fMRI-trace cache to a Hugging Face Hub **dataset**
   repo incrementally, so a Colab disconnect loses minutes of progress, not
   the whole run (ROADMAP.md's Colab-specific practicalities -- Colab's local
   disk does not survive a disconnect). This needs an HF token *with write
   access* -- set it as a Colab secret named `HF_TOKEN` (key icon in the left
   sidebar), or you'll be prompted to paste one. This is a different
   requirement from the gated-Llama token discussed for the (unrelated,
   optional) cross-modal retrieval spike in `vibe_retrieval.py` -- TRIBEv2
   itself is ungated, so no token is required just to load it; the token here
   is purely for **your own** Hub dataset repo to push the cache to.
3. If `music-brain` is a **private** GitHub repo, the plain `git clone` in the
   next cell will fail -- use the commented-out token variant instead.


## 1. Check GPU

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


## 2. Clone the repo

Edit `REPO_URL` if your GitHub username/repo differs.

In [ ]:
REPO_URL = "https://github.com/akshay-p-123/music-brain.git"

import os
if not os.path.isdir("music-brain"):
    # Private repo: use this instead (Colab secret named GITHUB_TOKEN), then
    # comment out the plain clone line above:
    # from google.colab import userdata
    # gh_token = userdata.get("GITHUB_TOKEN")
    # !git clone https://{gh_token}@github.com/akshay-p-123/music-brain.git
    !git clone {REPO_URL}
%cd music-brain


## 3. Install dependencies

`tribev2` is pinned to the exact commit this project was developed against
(reproducibility -- see `pip freeze` in the project's local venv); its own
`pyproject.toml` pulls in `neuralset`, `torch`, `transformers`, etc. for you.
The remaining lines cover `musicbrain`'s own extra requirements
(`pyproject.toml`) not already implied by `tribev2`.

In [ ]:
%pip install -q "tribev2 @ git+https://github.com/facebookresearch/tribev2.git@af58661791a351a448a489042a28f6c37e1c14b7"
%pip install -q nibabel nilearn hnswlib soundfile
%pip install -q -e .


## 4. Hugging Face auth (for pushing the cache)

Set an `HF_TOKEN` Colab secret (needs **write** access to create/push to a
dataset repo under your account) -- or leave `HUB_CACHE_REPO_ID = None`
below to skip Hub push entirely and keep the cache local-only for this
session (fine for a quick test, risky for the full 1802-clip run since a
disconnect then loses everything cached so far).

In [ ]:
import os

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = input("Paste an HF token with write access (or leave blank to skip Hub push): ").strip() or None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HF auth set.")
else:
    print("No HF_TOKEN -- Hub push/pull will be skipped, cache stays local-only this session.")


## 5. Config

Set `HUB_CACHE_REPO_ID` to a dataset repo under **your own** HF account
(created automatically on first push if it doesn't exist yet, via
`push_cache_to_hub`'s `create_repo(..., exist_ok=True)`).

In [ ]:
HUB_CACHE_REPO_ID = "your-hf-username/music-brain-deam-cache"  # or None to skip Hub push/pull
PUSH_EVERY = 50          # push to the Hub after every N newly-cached clips
WINDOW_S = 0.5           # ROADMAP.md Phase 1 item 6 -- starting point, not yet finalized
SEED = 0


## 6. Fetch DEAM + resume any previously-cached clips

`fetch_deam` downloads ~1.3GB (audio + annotations + metadata) if not
already present. `pull_cache_from_hub` restores any clips a previous,
disconnected session already pushed -- a no-op (returns 0) the first time
this repo is run, since the dataset repo won't exist yet.

In [ ]:
import sys
sys.path.insert(0, "src")

from musicbrain.datasets import deam
from musicbrain.fmri_cache import build_cache, pull_cache_from_hub, DEFAULT_CACHE_DIR

print("=== Fetching DEAM ===")
deam.fetch_deam()
va_by_song = deam.load_dynamic_va()
all_ids = deam.list_song_ids()
print(f"{len(all_ids)} DEAM clips available")

cache_dir = DEFAULT_CACHE_DIR  # data/cache/deam -- full-scale cache, distinct from the local smoke-test cache dir

if HUB_CACHE_REPO_ID:
    n_pulled = pull_cache_from_hub(HUB_CACHE_REPO_ID, cache_dir=cache_dir, token=HF_TOKEN)
    print(f"Resumed {n_pulled} previously-cached clip(s) from {HUB_CACHE_REPO_ID}")


## 7. Step 0 -- TRIBEv2 -> Schaefer-400 -> windowed cache, all clips

Full-length audio (no truncation, unlike the local smoke-test script) --
this is the long-running cell. `num_workers` auto-picks a parallel value on
Colab's Linux/fork runtime (see `tribev2_utils.load_tribev2_model`); pass
`num_workers=0` explicitly if you hit worker-related instability.

In [ ]:
audio_paths = {song_id: deam.deam_audio_path(song_id) for song_id in all_ids}

cache_results = build_cache(
    song_ids=all_ids,
    audio_paths=audio_paths,
    va_by_song=va_by_song,
    window_s=WINDOW_S,
    cache_dir=cache_dir,
    push_every=PUSH_EVERY if HUB_CACHE_REPO_ID else None,
    hub_repo_id=HUB_CACHE_REPO_ID,
    hub_token=HF_TOKEN,
)

import numpy as np
rates = [r.raw_trace_hz for r in cache_results if not np.isnan(r.raw_trace_hz)]
print(f"\nCached {len(cache_results)} clips. Mean raw TRIBEv2 rate: {np.mean(rates):.2f}Hz "
      f"(Phase 0/1 local validation both measured ~1.0Hz -- re-check ROADMAP.md item 6 "
      f"window-size decision against this real, full-scale number).")


## 8. Step 1 -- train trunk + VA head on the full cache

Split by clip (not window) so held-out correlation isn't leaked by seeing
another window from the same song.

In [ ]:
from musicbrain.train import WindowDataset, save_checkpoint, split_song_ids, train_step1

cached_ids = [r.song_id for r in cache_results]
train_ids, val_ids = split_song_ids(cached_ids, val_frac=0.2, seed=SEED)
print(f"train clips={len(train_ids)}  val clips={len(val_ids)}")

train_ds = WindowDataset(train_ids, cache_dir=cache_dir)
val_ds = WindowDataset(val_ids, cache_dir=cache_dir)
print(f"train windows={len(train_ds)}  val windows={len(val_ds)}")

model, history = train_step1(train_ds, val_ds, epochs=30, batch_size=256, seed=SEED)

ckpt_path = "external/cache/deam_full_trunk_va.pt"
save_checkpoint(model, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

if HUB_CACHE_REPO_ID:
    from huggingface_hub import HfApi
    HfApi(token=HF_TOKEN).upload_file(
        path_or_fileobj=ckpt_path,
        path_in_repo="deam_full_trunk_va.pt",
        repo_id=HUB_CACHE_REPO_ID,
        repo_type="dataset",
    )
    print(f"Pushed checkpoint to {HUB_CACHE_REPO_ID}")


## 9. Step 2 -- verification (PROJECT_SPEC.md Section 6 / ROADMAP.md exit criteria)

This is the real number: pooled held-out correlation *and* within-clip
dynamic tracking (the check that distinguishes "learned the trajectory"
from "learned the clip average").

In [ ]:
from musicbrain.verify import held_out_correlation, summarize_tracking, within_clip_tracking

corr = held_out_correlation(model, val_ds)
print(f"Held-out pooled correlation: valence r={corr.valence_r:.3f} arousal r={corr.arousal_r:.3f} (n={corr.n_windows} windows)")

tracking = within_clip_tracking(model, val_ds)
summary = summarize_tracking(tracking)
print(f"Within-clip tracking summary: {summary}")


## 10. Text branch + dynamic-vs-flat sanity check

Uses the vibe-descriptor anchor vocabulary (MTG-Jamendo mood/theme tags,
PROJECT_SPEC.md Section 3.3) -- this is the first time this sanity check
runs against a properly (not 15-clip-undertrained) trained model.

In [ ]:
from musicbrain.lexicon import NRCVAD
from musicbrain.soft_prompt import FrozenGenerationLLM
from musicbrain.text_branch import dynamic_vs_flat_sanity_check
from musicbrain.vibe_lexicon import build_vibe_anchor_set

lexicon = NRCVAD()
llm = FrozenGenerationLLM()
anchor_set = build_vibe_anchor_set(lexicon, llm)

sanity = dynamic_vs_flat_sanity_check(model, llm, anchor_set, val_ids, cache_dir=cache_dir)
print(f"Dynamic clip (song {sanity.dynamic_song_id}): {sanity.dynamic_sentence}")
print(f"  nearest-anchor fallback: {' -> '.join(sanity.dynamic_fallback)}")
print(f"Flat clip (song {sanity.flat_song_id}): {sanity.flat_sentence}")
print(f"  nearest-anchor fallback: {' -> '.join(sanity.flat_fallback)}")


## Next steps

- Compare Step 2's numbers against ROADMAP.md Phase 1's exit criteria
  ("actually correlated" within-clip tracking, not just clip-average
  correlated).
- Use the mean raw TRIBEv2 rate printed in Step 0 to settle the window-size
  decision (ROADMAP.md Phase 1 item 6) -- confirm it's still ~1Hz on real,
  full-length clips before locking in 0.5s windows vs. widening.
- If exit criteria are met, this checkpoint/cache is what Phase 2 builds on
  (scaling to the other five datasets).